# SAOU center-spin demo

Load one trained SAOU checkpoint, drive only the center spin along a circle, and compare the predicted local entropy-production increment with the homogeneous SAOU ground truth. The external pinning force is excluded from the ground truth, and the pinned center site is excluded from the metrics.

In [ ]:
import os
from pathlib import Path
import sys

import numpy as np
import torch
from tqdm.auto import trange


def find_repo_root():
    starts = (Path.cwd().resolve(), (Path.cwd() / 'KNEEP').resolve())
    for start in starts:
        for candidate in (start, *start.parents):
            if (candidate / 'models' / 'saou.py').is_file() and (candidate / 'shell_force.py').is_file():
                return candidate
    raise RuntimeError('Could not locate the KNEEP repository root.')


ROOT = find_repo_root()
matplotlib_config = ROOT / 'results' / '.matplotlib'
matplotlib_config.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(matplotlib_config))

import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.animation import FuncAnimation

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from models import saou
from models.saou import SAOUConfig
from shell_force import ShellForceKNEEP2D

# Edit only this path to select a trained model.
CHECKPOINT = ROOT / 'results' / 'saou_temperature' / 'checkpoints' / 'p01' / 'T_1' / 'repeat_001.pt'
OUTPUT = ROOT / 'results' / 'saou_center_spin' / 'center_spin_gt_vs_pred.mp4'
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

ROLLOUT_T = 1e-6
SOURCE_AMPLITUDE = 3.0
PIN_STRENGTH = 40.0
SOURCE_OMEGA = 0.5
BURN_STEPS = 10_000
N_TRANSITIONS = 10_000
SEED = 10_003
N_MOVIE_FRAMES = 240
INFERENCE_BATCH_SIZE = 64
FPS = 5


In [ ]:
if not CHECKPOINT.is_file():
    raise FileNotFoundError(f'Checkpoint not found: {CHECKPOINT}')

checkpoint = torch.load(CHECKPOINT, map_location='cpu', weights_only=True)
config = SAOUConfig(**checkpoint['saou_config'])
model = ShellForceKNEEP2D(**checkpoint['model_config']).to(DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
mean = checkpoint['mean'].to(device=DEVICE, dtype=torch.float32)
std = checkpoint['std'].to(device=DEVICE, dtype=torch.float32)

L = config.lattice_size
cy = cx = L // 2
operators = saou._build_operators(config)
generator = torch.Generator(device=DEVICE).manual_seed(SEED)
noise_scale = np.sqrt(2.0 * ROLLOUT_T * config.dt)


def source_target(time):
    phase = SOURCE_OMEGA * float(time)
    return torch.tensor(
        [SOURCE_AMPLITUDE * np.cos(phase), SOURCE_AMPLITUDE * np.sin(phase)],
        device=DEVICE,
        dtype=torch.float64,
    )


@torch.no_grad()
def advance(field, time):
    pin_force = torch.zeros_like(field)
    pin_force[cy, cx] = PIN_STRENGTH * (source_target(time) - field[cy, cx])
    noise = torch.randn(
        field.shape, generator=generator, device=DEVICE, dtype=field.dtype
    )
    return field + (saou._drift(field, operators, config) + pin_force) * config.dt + noise_scale * noise


@torch.no_grad()
def evaluate_pairs(pairs):
    predicted, ground_truth = [], []
    for start in range(0, len(pairs), INFERENCE_BATCH_SIZE):
        raw_pair = pairs[start:start + INFERENCE_BATCH_SIZE].to(DEVICE)
        normalized_pair = (raw_pair.float() - mean) / std
        raw_branch_maps = model(normalized_pair, return_maps=True)
        predicted.append((raw_branch_maps.sum(dim=1) / (L * L)).cpu())

        first = raw_pair[:, 0].permute(0, 2, 3, 1)
        second = raw_pair[:, 1].permute(0, 2, 3, 1)
        midpoint = 0.5 * (first + second)
        homogeneous_force = saou.irreversible_velocity(
            midpoint, operators, config.omega0
        ) / config.temperature
        ground_truth.append((homogeneous_force * (second - first)).sum(dim=-1).cpu())

    return torch.cat(predicted).numpy(), torch.cat(ground_truth).numpy()

print(f'checkpoint: {CHECKPOINT}')
print(f'device: {DEVICE}; training temperature: {config.temperature:g}')


In [ ]:
field = np.sqrt(ROLLOUT_T / config.gamma) * torch.randn(
    (L, L, 2), generator=generator, device=DEVICE, dtype=torch.float64
)

with torch.no_grad():
    for step in trange(BURN_STEPS, desc='center-spin burn-in'):
        field = advance(field, step * config.dt)

movie_ids = np.linspace(
    0, N_TRANSITIONS - 1, min(N_MOVIE_FRAMES, N_TRANSITIONS), dtype=int
)
selected_ids = set(movie_ids.tolist())
pairs, spin_frames = [], []
previous = field.detach().clone()

with torch.no_grad():
    for transition_id in trange(N_TRANSITIONS, desc='center-spin rollout'):
        field = advance(field, (BURN_STEPS + transition_id) * config.dt)
        current = field.detach().clone()
        if transition_id in selected_ids:
            pairs.append(torch.stack((previous.permute(2, 0, 1), current.permute(2, 0, 1))))
            spin_frames.append(current.cpu())
        previous = current

pairs = torch.stack(pairs)
spin_frames = torch.stack(spin_frames).numpy()
pred_ep_frames, true_ep_frames = evaluate_pairs(pairs)
spin_angles = np.arctan2(spin_frames[..., 1], spin_frames[..., 0])
diff_ep_frames = pred_ep_frames - true_ep_frames
movie_times = (BURN_STEPS + movie_ids + 1) * config.dt

center_mask = np.zeros((L, L), dtype=bool)
center_mask[cy, cx] = True
valid = ~center_mask
true_valid = true_ep_frames[:, valid].ravel()
pred_valid = pred_ep_frames[:, valid].ravel()
pearson_r = float(np.corrcoef(true_valid, pred_valid)[0, 1])
relative_rmse = float(
    np.linalg.norm(pred_valid - true_valid) / max(np.linalg.norm(true_valid), 1e-12)
)

print(
    f'homogeneous GT: T_train={config.temperature:g}; rollout_T={ROLLOUT_T:g}; '
    'external pin excluded; center site masked'
)
print(f'local-EP Pearson r={pearson_r:.4f}; relative RMSE={relative_rmse:.4f}')


In [ ]:
pooled_ep = np.concatenate((true_ep_frames[:, valid].ravel(), pred_ep_frames[:, valid].ravel()))
vmax_ep = max(1e-12, float(np.percentile(np.abs(pooled_ep), 99)))
vmax_diff = max(1e-12, float(np.percentile(np.abs(diff_ep_frames[:, valid]), 99)))
angle_cmap = plt.get_cmap('twilight').copy()
ep_cmap = plt.get_cmap('RdBu_r').copy()
ep_cmap.set_bad('lightgray')


def masked(frames, index):
    return np.ma.masked_where(center_mask, frames[index])


fig, axes = plt.subplots(1, 4, figsize=(16, 4.5), dpi=120)
spin_im = axes[0].imshow(
    spin_angles[0], origin='lower', cmap=angle_cmap, vmin=-np.pi, vmax=np.pi, interpolation='nearest'
)
true_im = axes[1].imshow(
    masked(true_ep_frames, 0), origin='lower', cmap=ep_cmap, vmin=-vmax_ep, vmax=vmax_ep, interpolation='nearest'
)
pred_im = axes[2].imshow(
    masked(pred_ep_frames, 0), origin='lower', cmap=ep_cmap, vmin=-vmax_ep, vmax=vmax_ep, interpolation='nearest'
)
diff_im = axes[3].imshow(
    masked(diff_ep_frames, 0), origin='lower', cmap=ep_cmap, vmin=-vmax_diff, vmax=vmax_diff, interpolation='nearest'
)

axes[0].set_title(r'Spin profile $\theta(x,y)$')
axes[1].set_title('Homogeneous-theory true local EP')
axes[2].set_title('Predicted local EP')
axes[3].set_title('Prediction - true')
axes[0].scatter([cx], [cy], s=60, facecolors='none', edgecolors='white', linewidths=1.2)
for axis in axes:
    axis.set_xticks([])
    axis.set_yticks([])
fig.colorbar(spin_im, ax=axes[0], fraction=0.046, label='spin angle')
fig.colorbar(true_im, ax=axes[1:3], fraction=0.026, pad=0.03, label='local EP increment')
fig.colorbar(diff_im, ax=axes[3], fraction=0.046, label='prediction - true')
title = fig.suptitle('')


def update(frame):
    spin_im.set_data(spin_angles[frame])
    true_im.set_data(masked(true_ep_frames, frame))
    pred_im.set_data(masked(pred_ep_frames, frame))
    diff_im.set_data(masked(diff_ep_frames, frame))
    title.set_text(
        f'Center spin: omega={SOURCE_OMEGA:g}, t={movie_times[frame]:.3f} | '
        f'EP r={pearson_r:.3f}, relative RMSE={relative_rmse:.3f}'
    )
    return spin_im, true_im, pred_im, diff_im, title


movie = FuncAnimation(
    fig, update, frames=len(movie_ids), interval=1000 / FPS, blit=False
)
fig.subplots_adjust(left=0.03, right=0.97, bottom=0.08, top=0.84, wspace=0.35)

if not animation.writers.is_available('ffmpeg'):
    raise RuntimeError('ffmpeg is required to save the MP4.')
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
movie.save(
    OUTPUT,
    writer=animation.FFMpegWriter(fps=FPS, bitrate=2400, extra_args=['-pix_fmt', 'yuv420p']),
    dpi=120,
)
if not OUTPUT.is_file() or OUTPUT.stat().st_size == 0:
    raise RuntimeError(f'MP4 was not created correctly: {OUTPUT}')
print(f'saved: {OUTPUT} ({OUTPUT.stat().st_size / 1e6:.1f} MB)')
plt.show()
